<div style="padding: 20px; background-color: #f0f8ff; border-radius: 10px; border-left: 8px solid #0078d4;">
    <h1 style="color: #0078d4; margin-bottom: 5px;">📘 Azure Machine Learning Toolkit</h1>
    <p><i>This notebook centralizes useful code snippets for working with the <b>azure-ai-ml</b> library.</i></p>
</div>

In [ ]:
%pip install azure-ai-ml azure-identity

# Connect to Azure ml

In [ ]:
# subscription_id : your Azure subscription ID
# resource_group  : name of the resource group containing the workspace
# workspace_name  : name of the Azure ML workspace

subscription_id = "<SUBSCRIPTION_ID>"
resource_group = "<RESOURCE_GROUP>"
workspace_name = "<AML_WORKSPACE_NAME>"

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient(
    DefaultAzureCredential(), subscription_id, resource_group, workspace_name
)

## Other authentication methods

In [ ]:
# Connect using a config file (config.json downloaded from the Azure ML portal)
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(credential=DefaultAzureCredential())


# Interactive authentication (opens a browser sign-in window)
from azure.identity import InteractiveBrowserCredential

credential = InteractiveBrowserCredential()


# Service Principal authentication (CI/CD, non-interactive scripts)
from azure.identity import ClientSecretCredential

credential = ClientSecretCredential(
    tenant_id="<TENANT_ID>",
    client_id="<CLIENT_ID>",
    client_secret="<CLIENT_SECRET>",
)


# Managed identity authentication (from an Azure VM/compute)
from azure.identity import ManagedIdentityCredential

credential = ManagedIdentityCredential(client_id="<MANAGED_IDENTITY_CLIENT_ID>")  # client_id is optional for the system-assigned identity

ml_client = MLClient(credential, subscription_id, resource_group, workspace_name)


# Retrieve a secret from the Key Vault linked to the workspace (pip install azure-keyvault-secrets)
from azure.keyvault.secrets import SecretClient

secret_client = SecretClient(
    vault_url="https://<keyvault-name>.vault.azure.net",
    credential=DefaultAzureCredential(),
)
db_password = secret_client.get_secret("db-password").value

# Create a space work

In [ ]:
from azure.ai.ml.entities import Workspace

workspace_name = "mlw-example"

ws_basic = Workspace(
    name=workspace_name,
    location="eastus",
    display_name="Basic workspace-example",
    description="This example shows how to create a basic workspace",
)
ml_client.workspaces.begin_create(ws_basic)

## List / Get / Delete a workspace

In [ ]:
# List all workspaces in the resource group
for ws in ml_client.workspaces.list():
    print(ws.name)

# Get a specific workspace
ws = ml_client.workspaces.get(name=workspace_name)
print(ws.location, ws.display_name)

# Delete a workspace (and optionally its dependent resources)
ml_client.workspaces.begin_delete(
    name=workspace_name, delete_dependent_resources=True
)

## Submit a job


In [ ]:
from azure.ai.ml import command


job = command(
    code="./src",
    command="python diabetes-training.py",
    environment="AzureML-sklearn-1.5@latest",
    compute="aml-cluster",
    display_name="diabetes-pythonv2-train",
    experiment_name="diabetes-training"
)

# submit job
returned_job = ml_client.create_or_update(job)
aml_url = returned_job.studio_url
print("Monitor your job at", aml_url)

## Monitor, list and manage jobs

In [ ]:
# Stream the logs of a job (blocks until the job finishes)
ml_client.jobs.stream(returned_job.name)

# Get the current status of a job
job_info = ml_client.jobs.get(returned_job.name)
print(job_info.status)

# Download the outputs/logs of a completed job
ml_client.jobs.download(name=returned_job.name, download_path="./outputs")

# List jobs (most recent first), optionally filter by experiment
for j in ml_client.jobs.list(experiment_name="diabetes-training"):
    print(j.name, j.status)

# Cancel a running job
ml_client.jobs.begin_cancel(returned_job.name).result()

# Example training script (train.py)

In [ ]:
# train.py - referenced by the command() jobs above, e.g.:
# command="python train.py --training_data ${{inputs.training_data}} --reg_rate ${{inputs.reg_rate}}"

import argparse

import pandas as pd
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--training_data", type=str, help="path to the training data")
    parser.add_argument("--reg_rate", type=float, default=0.01, help="regularization rate")
    return parser.parse_args()


def main(args):
    # mlflow.autolog() automatically logs params, metrics and the trained model
    mlflow.autolog()

    df = pd.read_csv(args.training_data)
    X = df.drop(columns=["Diabetic"])
    y = df["Diabetic"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Log a parameter explicitly (in addition to, or instead of, autolog)
    mlflow.log_param("reg_rate", args.reg_rate)

    model = LogisticRegression(C=1 / args.reg_rate, max_iter=1000)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("AUC", roc_auc_score(y_test, y_proba))

    # Log any file as an artifact (plot, report, etc.)
    with open("summary.txt", "w") as f:
        f.write(f"Trained on {len(X_train)} rows, tested on {len(X_test)} rows.")
    mlflow.log_artifact("summary.txt")


if __name__ == "__main__":
    main(parse_args())

In [ ]:
# Corresponding job submission (./src/train.py is the script defined above)
from azure.ai.ml import command, Input
from azure.ai.ml.constants import AssetTypes

job = command(
    code="./src",
    command="python train.py --training_data ${{inputs.training_data}} --reg_rate ${{inputs.reg_rate}}",
    inputs={
        "training_data": Input(type=AssetTypes.URI_FILE, path="azureml:diabetes-data:1"),
        "reg_rate": 0.01,
    },
    environment="AzureML-sklearn-1.5@latest",
    compute="aml-cluster",
    display_name="diabetes-train",
    experiment_name="diabetes-training",
)

returned_job = ml_client.create_or_update(job)

# Create a Data Store (blob)

## without SAS

In [ ]:
from azure.ai.ml.entities import AzureBlobDatastore, AccountKeyConfiguration

blob_datastore = AzureBlobDatastore(
    name="blob_example",
    description="Datastore pointing to a blob container",
    account_name="mytestblobstore",
    container_name="data-container",
    credentials=AccountKeyConfiguration(
        account_key="XXXxxxXXXxXXXXxxXXX"
    ),
)
ml_client.create_or_update(blob_datastore)

## with SAS

In [ ]:
from azure.ai.ml.entities import AzureBlobDatastore, SasTokenConfiguration

blob_datastore = AzureBlobDatastore(
    name="blob_sas_example",
    description="Datastore pointing to a blob container",
    account_name="mytestblobstore",
    container_name="data-container",
    credentials=SasTokenConfiguration(
        sas_token="?xx=XXXX-XX-XX&xx=xxxx&xxx=xxx&xx=xxxxxxxxxxx&xx=XXXX-XX-XXXXX:XX:XXX&xx=XXXX-XX-XXXXX:XX:XXX&xxx=xxxxx&xxx=XXxXXXxxxxxXXXXXXXxXxxxXXXXXxxXXXXXxXXXXxXXXxXXxXX"
    ),
)
ml_client.create_or_update(blob_datastore)

## List / Get / Delete a datastore

In [ ]:
# List all datastores
for ds in ml_client.datastores.list():
    print(ds.name, ds.type)

# Get a specific datastore
ds = ml_client.datastores.get(name="blob_example")

# Get the workspace's default datastore
default_ds = ml_client.datastores.get_default()
print(default_ds.name)

# Delete a datastore
ml_client.datastores.delete(name="blob_example")

# Create a URI file (folder) ressource

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

my_path = '<supported-path>'  # local path, "azureml://datastores/<ds>/paths/<path>", "https://...", "abfss://..."

my_data = Data(
    path=my_path,
    type=AssetTypes.URI_FILE,   # <-- AssetTypes.URI_FOLDER for a folder
    description="<description>",
    name="<name>",
    version="<version>"
)

ml_client.data.create_or_update(my_data)

## List / Get versions / Archive a data asset

In [ ]:
# List all data assets registered in the workspace
for d in ml_client.data.list():
    print(d.name)

# List all versions of a data asset
for version in ml_client.data.list(name="<name>"):
    print(version.version)

# Get a specific version
data_asset = ml_client.data.get(name="<name>", version="1")
print(data_asset.path)

# Archive / restore a version (archived versions are hidden but not deleted)
ml_client.data.archive(name="<name>", version="1")
ml_client.data.restore(name="<name>", version="1")

# Create a computing instance

In [ ]:
from azure.ai.ml.entities import ComputeInstance

ci_basic_name = "basic-ci-12345"
ci_basic = ComputeInstance(
    name=ci_basic_name, 
    size="STANDARD_DS3_v2"
)
ml_client.begin_create_or_update(ci_basic).result()

## List / Get / Start / Stop / Delete compute

In [ ]:
# List all compute resources (instances and clusters)
for compute in ml_client.compute.list():
    print(compute.name, compute.type, compute.provisioning_state)

# List available VM sizes
for size in ml_client.compute.list_sizes():
    print(size.name, size.family, size.v_cp_us, size.gpus)

# Get a specific compute
ci = ml_client.compute.get(ci_basic_name)

# Stop / start a compute instance (to control cost)
ml_client.compute.begin_stop(ci_basic_name).result()
ml_client.compute.begin_start(ci_basic_name).result()

# Delete a compute instance or cluster
ml_client.compute.begin_delete(ci_basic_name).result()

# Create a compute cluster

In [ ]:
from azure.ai.ml.entities import AmlCompute

cluster_basic = AmlCompute(
    name="cpu-cluster",
    type="amlcompute",
    size="STANDARD_DS3_v2",
    location="westus",
    min_instances=0,
    max_instances=2,
    idle_time_before_scale_down=120,
    tier="low_priority",
)
ml_client.begin_create_or_update(cluster_basic).result()


# size: specifies the VM size of each node in the compute cluster. This depends on the VM sizes
# available in Azure. Besides the size, you can also specify whether you want to use CPUs or GPUs.
# max_instances: specifies the maximum number of nodes your compute cluster can scale out to. The
# number of parallel workloads your cluster can handle is comparable to the number of nodes it can
# scale to.
# tier: specifies whether your VMs are low priority or dedicated. Setting low priority can reduce
# costs, as availability is not guaranteed.

# Use a computer cluster

In [ ]:
from azure.ai.ml import command

# configure job
job = command(
    code="./src",
    command="python diabetes-training.py",
    environment="AzureML-sklearn-0.24-ubuntu18.04-py37-cpu@latest",
    compute="cpu-cluster",
    display_name="train-with-cluster",
    experiment_name="diabetes-training"
    )

# submit job
returned_job = ml_client.create_or_update(job)
aml_url = returned_job.studio_url
print("Monitor your job at", aml_url)

# Distributed training (multi-node / multi-GPU)

In [ ]:
from azure.ai.ml import command

# PyTorch DistributedDataParallel: instance_count = number of nodes,
# process_count_per_instance = number of GPUs/processes per node
job_pytorch = command(
    code="./src",
    command="python train.py --epochs 10",
    environment="AzureML-pytorch-2.2-cuda12.1@latest",
    compute="gpu-cluster",
    instance_count=2,
    distribution={
        "type": "PyTorch",
        "process_count_per_instance": 4,
    },
    display_name="distributed-pytorch-job",
    experiment_name="distributed-training",
)

# MPI distribution (e.g. Horovod)
job_mpi = command(
    code="./src",
    command="python train.py",
    environment="AzureML-pytorch-2.2-cuda12.1@latest",
    compute="gpu-cluster",
    instance_count=2,
    distribution={
        "type": "Mpi",
        "process_count_per_instance": 4,
    },
)

# TensorFlow distribution (parameter server / worker)
job_tf = command(
    code="./src",
    command="python train.py",
    environment="AzureML-tensorflow-2.16-cuda12@latest",
    compute="gpu-cluster",
    instance_count=2,
    distribution={
        "type": "TensorFlow",
        "worker_count": 2,
    },
)

returned_job = ml_client.create_or_update(job_pytorch)

# Env

In [6]:
# see environement available
envs = ml_client.environments.list()
for env in envs:
    print(env.name)

# Specific environement detail
env = ml_client.environments.get(name="my-environment", version="1")
print(env)


# create specialized env from a docker image

In [ ]:
from azure.ai.ml.entities import Environment

env_docker_image = Environment(
    image="pytorch/pytorch:latest",
    #conda_file, ---> from conda (need A conda specification file (YAML)
    name="public-docker-image-example",
    description="Environment created from a public Docker image.",
)
ml_client.environments.create_or_update(env_docker_image)

## Create an environment from a Conda specification file

In [ ]:
from azure.ai.ml.entities import Environment

env_conda = Environment(
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04",
    conda_file="./conda.yml",
    name="training-environment",
    description="Environment created from a curated Docker image plus a Conda spec file.",
)
ml_client.environments.create_or_update(env_conda)

# Example conda.yml content:
# name: training-env
# channels:
#   - conda-forge
# dependencies:
#   - python=3.10
#   - pip
#   - pip:
#       - mlflow
#       - azureml-mlflow
#       - scikit-learn
#       - pandas

# Create a data ressource MLTable 

In [ ]:
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import Input

my_training_data_input = Input(type=AssetTypes.MLTABLE, path="azureml:input-data-automl:1")

# configurate a AutoML job


In [ ]:
from azure.ai.ml import automl

# configure the classification job
classification_job = automl.classification(
    compute="aml-cluster",
    experiment_name="auto-ml-class-dev",
    training_data=my_training_data_input,
    target_column_name="Diabetic",
    primary_metric="accuracy", # <-- 'AUCWeighted', 'Accuracy', NormMacroRecall', 'AveragePrecisionScoreWeighted', 'PrecisionScoreWeighted'
    n_cross_validations=5,
    enable_model_explainability=True
)

# Set limit

classification_job.set_limits(
    timeout_minutes=60, 
    trial_timeout_minutes=20, 
    max_trials=5,
    enable_early_termination=True #<-- indicates whether the experiment should be terminated if the score does not improve in the short term.
)

# submit the AutoML job
returned_job = ml_client.jobs.create_or_update(
    classification_job
) 


# MLflow


In [ ]:
%pip install mlflow azureml-mlflow

import mlflow

# Retrieve the tracking URI from the workspace (or copy it from the Azure ML portal)
mlflow_tracking_uri = ml_client.workspaces.get(workspace_name).mlflow_tracking_uri
mlflow.set_tracking_uri(mlflow_tracking_uri)

## create a MLworkflow experiment (Autolog)

In [ ]:
mlflow.set_experiment(experiment_name="heart-condition-classifier")

from xgboost import XGBClassifier # <-- example

with mlflow.start_run():
    mlflow.xgboost.autolog()

    model = XGBClassifier(use_label_encoder=False, eval_metric="logloss")
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

## create a MLworkflow experiment (personalized)

In [ ]:
mlflow.set_experiment(experiment_name="heart-condition-classifier")

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

with mlflow.start_run():
    model = XGBClassifier(use_label_encoder=False, eval_metric="logloss")
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", accuracy)

## Search experiences

In [ ]:
#all experiences 
experiments = mlflow.search_experiments(max_results=2)
for exp in experiments:
    print(exp.name)

#specific experience
exp = mlflow.get_experiment_by_name(experiment_name)
print(exp)


# if you want to search for achived experiment

from mlflow.entities import ViewType

experiments = mlflow.search_experiments(view_type=ViewType.ALL)
for exp in experiments:
    print(exp.name)

## Search run

In [ ]:
mlflow.search_runs(exp.experiment_id, order_by=["start_time DESC"], max_results=2) #<- by default start_time is desc


# You can also search for an execution with a specific combination in the hyperparameters:
mlflow.search_runs(
    exp.experiment_id, filter_string="params.num_boost_round='100'", max_results=2
)


# Define a search space

In [ ]:
from azure.ai.ml.sweep import Choice, Normal

command_job_for_sweep = job(
    batch_size=Choice(values=[16, 32, 64]),    
    learning_rate=Normal(mu=10, sigma=3),
)

# Grid sampling

In [ ]:
from azure.ai.ml.sweep import Choice

command_job_for_sweep = command_job(
    batch_size=Choice(values=[16, 32, 64]),
    learning_rate=Choice(values=[0.01, 0.1, 1.0]),
)

sweep_job = command_job_for_sweep.sweep(
    sampling_algorithm = "grid",
    ...
)

# Random sampling

In [ ]:
from azure.ai.ml.sweep import Normal, Uniform

command_job_for_sweep = command_job(
    batch_size=Choice(values=[16, 32, 64]),   
    learning_rate=Normal(mu=10, sigma=3),
)

sweep_job = command_job_for_sweep.sweep(
    sampling_algorithm = "random",
    ...
)

# with sobol

from azure.ai.ml.sweep import RandomSamplingAlgorithm

sweep_job = command_job_for_sweep.sweep(
    sampling_algorithm = RandomSamplingAlgorithm(seed=123, rule="sobol"),
    ...
)

# Bayes sampling

In [ ]:
from azure.ai.ml.sweep import Uniform, Choice

command_job_for_sweep = job(
    batch_size=Choice(values=[16, 32, 64]),    
    learning_rate=Uniform(min_value=0.05, max_value=0.1),
)

sweep_job = command_job_for_sweep.sweep(
    sampling_algorithm = "bayesian",
    ...
)

# configurate a anticipate stop


## Bandit strategy

In [ ]:
from azure.ai.ml.sweep import BanditPolicy

sweep_job.early_termination = BanditPolicy(
    slack_amount = 0.2, 
    delay_evaluation = 5, 
    evaluation_interval = 1
)

## Median stop strategy

In [ ]:
from azure.ai.ml.sweep import MedianStoppingPolicy

sweep_job.early_termination = MedianStoppingPolicy(
    delay_evaluation = 5, 
    evaluation_interval = 1
)

## Truncation selection strategy

In [ ]:
from azure.ai.ml.sweep import TruncationSelectionPolicy

sweep_job.early_termination = TruncationSelectionPolicy(
    evaluation_interval=1, 
    truncation_percentage=20, 
    delay_evaluation=4 
)

# Configurate and execute a scanning task

In [5]:
from azure.ai.ml import command

# configure command job as base
job = command(
    code="./src",
    command="python train.py --regularization ${{inputs.reg_rate}}",
    inputs={
        "reg_rate": 0.01,
    },
    environment="AzureML-sklearn-0.24-ubuntu18.04-py37-cpu@latest",
    compute="aml-cluster",
    )

#You can then replace your input parameters with your search space:
from azure.ai.ml.sweep import Choice

command_job_for_sweep = job(
    reg_rate=Choice(values=[0.01, 0.1, 1]),
)

# Finally, call sweep() on your command job to traverse your search space:
from azure.ai.ml import MLClient

# apply the sweep parameter to obtain the sweep_job
sweep_job = command_job_for_sweep.sweep(
    compute="aml-cluster",
    sampling_algorithm="grid",
    primary_metric="Accuracy",
    goal="Maximize",
)

# set the name of the sweep job experiment
sweep_job.experiment_name="sweep-example"

# define the limits for this sweep
sweep_job.set_limits(max_total_trials=4, max_concurrent_trials=2, timeout=7200)

# submit the sweep
returned_sweep_job = ml_client.create_or_update(sweep_job)


## Retrieve the best trial from a sweep job

In [ ]:
# Wait for the sweep job to complete
ml_client.jobs.stream(returned_sweep_job.name)

sweep_job = ml_client.jobs.get(returned_sweep_job.name)

# The id of the best performing trial is exposed in the job properties
best_run_id = sweep_job.properties["best_child_run_id"]
print("Best run:", best_run_id)

best_run = ml_client.jobs.get(best_run_id)
print(best_run.properties)

# Download the outputs (e.g. the model) of the best trial
ml_client.jobs.download(name=best_run_id, download_path="./best_model", output_name="model")

# Register the best trial's model directly from its job output
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

best_model = Model(
    path=f"azureml://jobs/{best_run_id}/outputs/artifacts/paths/model/",
    name="sweep-best-model",
    type=AssetTypes.MLFLOW_MODEL,
)
ml_client.models.create_or_update(best_model)

# Register a component 

In [ ]:
from azure.ai.ml import load_component

parent_dir = ""

loaded_component_prep = load_component(source=parent_dir + "./prep.yml")

prep = ml_client.components.create_or_update(loaded_component_prep)

## Example component YAML (prep.yml)

```yaml
$schema: https://azuremlschemas.azureedge.net/latest/commandComponent.schema.json
name: prep_data
display_name: Prepare training data
version: 1
type: command
inputs:
  input_data:
    type: uri_folder
outputs:
  output_data:
    type: uri_folder
code: ./prep_src
environment: azureml:AzureML-sklearn-1.5@latest
command: >-
  python prep.py
  --input_data ${{inputs.input_data}}
  --output_data ${{outputs.output_data}}
```

# Create a pipeline


In [2]:
from azure.ai.ml.dsl import pipeline

@pipeline()
def pipeline_function_name(pipeline_job_input):
    prep_data = loaded_component_prep(input_data=pipeline_job_input)
    train_model = loaded_component_train(training_data=prep_data.outputs.output_data)

    return {
        "pipeline_job_transformed_data": prep_data.outputs.output_data,
        "pipeline_job_trained_model": train_model.outputs.model_output,
    }


# To pass a registered data asset as input to the pipeline job, call the function you
# created with the data asset as input:

from azure.ai.ml import Input
from azure.ai.ml.constants import AssetTypes

pipeline_job = pipeline_function_name(
    Input(type=AssetTypes.URI_FILE,
    path="azureml:data:1"
))

# Configurate a pipeline work


In [ ]:
# change the output mode
pipeline_job.outputs.pipeline_job_transformed_data.mode = "upload"
pipeline_job.outputs.pipeline_job_trained_model.mode = "upload"


# set pipeline level compute
pipeline_job.settings.default_compute = "aml-cluster"

# set pipeline level datastore
pipeline_job.settings.default_datastore = "workspaceblobstore"

print(pipeline_job)

# Execute a pipeline work

In [ ]:
# submit job to workspace
pipeline_job = ml_client.jobs.create_or_update(
    pipeline_job, experiment_name="pipeline_job"
)

# Plan a pipeline work

In [ ]:
# create the schedule
from azure.ai.ml.entities import RecurrenceTrigger

schedule_name = "run_every_minute"

recurrence_trigger = RecurrenceTrigger(
    frequency="minute",
    interval=1,
)

# Schedule
from azure.ai.ml.entities import JobSchedule

job_schedule = JobSchedule(
    name=schedule_name, trigger=recurrence_trigger, create_job=pipeline_job
)

job_schedule = ml_client.schedules.begin_create_or_update(
    schedule=job_schedule
).result()

# Delete the schedule

ml_client.schedules.begin_disable(name=schedule_name).result()
ml_client.schedules.begin_delete(name=schedule_name).result()

# Personalize signature MLmodel

In [ ]:
import pandas as pd
from sklearn import datasets
from sklearn.ensemble import RandomForestClassifier
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

iris = datasets.load_iris()
iris_train = pd.DataFrame(iris.data, columns=iris.feature_names)
clf = RandomForestClassifier(max_depth=7, random_state=0)
clf.fit(iris_train, iris.target)

# Infer the signature from the training dataset and model's predictions
signature = infer_signature(iris_train, clf.predict(iris_train))

# Log the scikit-learn model with the custom signature
mlflow.sklearn.log_model(clf, "iris_rf", signature=signature)

####create the signature manually
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

# Define the schema for the input data
input_schema = Schema([
  ColSpec("double", "sepal length (cm)"),
  ColSpec("double", "sepal width (cm)"),
  ColSpec("double", "petal length (cm)"),
  ColSpec("double", "petal width (cm)"),
])

# Define the schema for the output data
output_schema = Schema([ColSpec("long")])

# Create the signature object
signature = ModelSignature(inputs=input_schema, outputs=output_schema)


# RAI Insights dashboard constructor

In [ ]:
# ml_client_registry : client connected to the registry that contains the RAI components
# ml_client_registry = MLClient(credential, registry_name="azureml")
#
# expected_model_id / azureml_model_id : identifier of the registered MLflow model,
# e.g. "azureml:<model_name>:<version>"

rai_constructor_component = ml_client_registry.components.get(
    name="microsoft_azureml_rai_tabular_insight_constructor", label="latest")

#Add Explanation to RAI Insights dashboard component:
rai_explanation_component = ml_client_registry.components.get(
    name="microsoft_azureml_rai_tabular_explanation", label="latest")

#Gather RAI Insights dashboard
rai_gather_component = ml_client_registry.components.get(
    name="microsoft_azureml_rai_tabular_insight_gather", label="latest")

#Generate the pipeline

from azure.ai.ml import Input, dsl
from azure.ai.ml.constants import AssetTypes

@dsl.pipeline(
    compute="aml-cluster",
    experiment_name="Create RAI Dashboard",
)
def rai_decision_pipeline(
    target_column_name, train_data, test_data
):
    # Initiate the RAIInsights
    create_rai_job = rai_constructor_component(
        title="RAI dashboard diabetes",
        task_type="classification",
        model_info=expected_model_id,
        model_input=Input(type=AssetTypes.MLFLOW_MODEL, path=azureml_model_id),
        train_dataset=train_data,
        test_dataset=test_data,
        target_column_name="Predictions",
    )
    create_rai_job.set_limits(timeout=30)

    # Add explanations
    explanation_job = rai_explanation_component(
        rai_insights_dashboard=create_rai_job.outputs.rai_insights_dashboard,
        comment="add explanation",
    )
    explanation_job.set_limits(timeout=10)

    # Combine everything
    rai_gather_job = rai_gather_component(
        constructor=create_rai_job.outputs.rai_insights_dashboard,
        insight=explanation_job.outputs.explanation,
    )
    rai_gather_job.set_limits(timeout=10)

    rai_gather_job.outputs.dashboard.mode = "upload"

    return {
        "dashboard": rai_gather_job.outputs.dashboard,
    }

# Create an endpoint


In [ ]:
# name: Name of the endpoint. Must be unique within the Azure region.
# auth_mode: Use key for key-based authentication. Use aml_token for Azure Machine Learning token-based authentication.

from azure.ai.ml.entities import ManagedOnlineEndpoint

# create an online endpoint
endpoint = ManagedOnlineEndpoint(
    name="endpoint-example",
    description="Online endpoint",
    auth_mode="key",
)

ml_client.begin_create_or_update(endpoint).result()

# Deploy an MLflow model to an endpoint

In [ ]:
from azure.ai.ml.entities import Model, ManagedOnlineDeployment
from azure.ai.ml.constants import AssetTypes

# create a blue deployment
model = Model(
    path="./model",
    type=AssetTypes.MLFLOW_MODEL,
    description="my sample mlflow model",
)

blue_deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name="endpoint-example",
    model=model,
    instance_type="Standard_F4s_v2",
    instance_count=1,
)

ml_client.online_deployments.begin_create_or_update(blue_deployment).result()

In [ ]:
# blue deployment takes 100 traffic
endpoint.traffic = {"blue": 100}
ml_client.begin_create_or_update(endpoint).result()

In [1]:
#to delete 
ml_client.online_endpoints.begin_delete(name="endpoint-example")

## Test, list and manage a real-time endpoint

In [ ]:
# Test the endpoint with sample data (sample-request.json contains the input payload)
result = ml_client.online_endpoints.invoke(
    endpoint_name="endpoint-example",
    deployment_name="blue",  # optional: target a specific deployment
    request_file="./sample-request.json",
)
print(result)

# Get endpoint details (scoring URI) and authentication keys
endpoint = ml_client.online_endpoints.get(name="endpoint-example")
print(endpoint.scoring_uri)

keys = ml_client.online_endpoints.get_keys(name="endpoint-example")
print(keys.primary_key)

# List endpoints and their deployments
for ep in ml_client.online_endpoints.list():
    print(ep.name)

for dep in ml_client.online_deployments.list(endpoint_name="endpoint-example"):
    print(dep.name)

# Get deployment logs (useful to debug the scoring script)
logs = ml_client.online_deployments.get_logs(
    name="blue", endpoint_name="endpoint-example", lines=100
)
print(logs)

# Delete a single deployment without deleting the endpoint
ml_client.online_deployments.begin_delete(
    name="blue", endpoint_name="endpoint-example"
).result()

# Create the scoring script

In [ ]:
#The scoring script must include two functions:

# init(): called when the service is initialized.
#run(): called when new data is sent to the service.

import json
import joblib
import numpy as np
import os

# called when the deployment is created or updated
def init():
    global model
    # get the path to the registered model file and load it
    model_path = os.path.join(os.getenv('AZUREML_MODEL_DIR'), 'model.pkl')
    model = joblib.load(model_path)

# called when a request is received
def run(raw_data):
    # get the input data as a numpy array
    data = np.array(json.loads(raw_data)['data'])
    # get a prediction from the model
    predictions = model.predict(data)
    # return the predictions as any JSON serializable format
    return predictions.tolist()

# Create an environment

In [ ]:
from azure.ai.ml.entities import Environment

env = Environment(
    image="mcr.microsoft.com/azureml/openmpi3.1.2-ubuntu18.04",
    conda_file="./src/conda.yml",
    name="deployment-environment",
    description="Environment created from a Docker image plus Conda environment.",
)
ml_client.environments.create_or_update(env)

# Create deployment

In [ ]:
# To deploy a model to an endpoint, you can specify the compute configuration with two parameters:
# - instance_type: the size of the virtual machine to use.
# - instance_count: the number of instances to use.

from azure.ai.ml.entities import Model, ManagedOnlineDeployment, CodeConfiguration
from azure.ai.ml.constants import AssetTypes

model = Model(path="./model", type=AssetTypes.CUSTOM_MODEL)

blue_deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name="endpoint-example",
    model=model,
    environment="deployment-environment",
    code_configuration=CodeConfiguration(
        code="./src", scoring_script="score.py"
    ),
    instance_type="Standard_DS2_v2",
    instance_count=1,
)

ml_client.online_deployments.begin_create_or_update(blue_deployment).result()

## Create a batch processing endpoint

In [ ]:
from azure.ai.ml.entities import BatchEndpoint

# create a batch endpoint
endpoint = BatchEndpoint(
    name="endpoint-example",
    description="A batch endpoint",
)

ml_client.batch_endpoints.begin_create_or_update(endpoint)

## Use compute clusters for batch deployments

In [ ]:
from azure.ai.ml.entities import AmlCompute

cpu_cluster = AmlCompute(
    name="aml-cluster",
    type="amlcompute",
    size="STANDARD_DS11_V2",
    min_instances=0,
    max_instances=4,
    idle_time_before_scale_down=120,
    tier="Dedicated",
)

cpu_cluster = ml_client.compute.begin_create_or_update(cpu_cluster)

## Register an MLflow model

In [ ]:
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

model_name = 'mlflow-model'
model = ml_client.models.create_or_update(
    Model(name=model_name, path='./model', type=AssetTypes.MLFLOW_MODEL)
)

## List / Get / Download / Archive a model

In [ ]:
# List all registered models
for m in ml_client.models.list():
    print(m.name)

# List all versions of a model
for v in ml_client.models.list(name=model_name):
    print(v.version)

# Get a specific model version
model_v1 = ml_client.models.get(name=model_name, version="1")
print(model_v1.path)

# Download a model's files locally
ml_client.models.download(name=model_name, version="1", download_path="./downloaded_model")

# Archive / restore a model version
ml_client.models.archive(name=model_name, version="1")
ml_client.models.restore(name=model_name, version="1")

## Deploy an MLflow model to an endpoint


In [ ]:
# When configuring the model deployment, you can specify:

# instance_count: number of compute nodes to use to generate predictions.
# max_concurrency_per_instance: maximum number of parallel scoring script executions per compute node.
# mini_batch_size: number of files per scoring script execution.
# output_action: What to do with predictions: summary_only or append_row.
# output_file_name: File to which predictions will be added, if you choose append_row for output_action.

from azure.ai.ml.entities import BatchDeployment, BatchRetrySettings
from azure.ai.ml.constants import BatchDeploymentOutputAction

deployment = BatchDeployment(
    name="forecast-mlflow",
    description="A sales forecaster",
    endpoint_name=endpoint.name,
    model=model,
    compute="aml-cluster",
    instance_count=2,
    max_concurrency_per_instance=2,
    mini_batch_size=2,
    output_action=BatchDeploymentOutputAction.APPEND_ROW,
    output_file_name="predictions.csv",
    retry_settings=BatchRetrySettings(max_retries=3, timeout=300),
    logging_level="info",
)
ml_client.batch_deployments.begin_create_or_update(deployment)


# Trigger batch scoring


In [ ]:
from azure.ai.ml import Input
from azure.ai.ml.constants import AssetTypes

input_data = Input(type=AssetTypes.URI_FOLDER, path="azureml:new-data:1")

job = ml_client.batch_endpoints.invoke(
    endpoint_name=endpoint.name,
    input=input_data)

## Monitor and manage batch endpoints

In [ ]:
# Stream the logs / wait for the batch scoring job to complete
ml_client.jobs.stream(job.name)

# List batch endpoints and their deployments
for ep in ml_client.batch_endpoints.list():
    print(ep.name)

for dep in ml_client.batch_deployments.list(endpoint_name=endpoint.name):
    print(dep.name)

# Delete a batch endpoint (and its deployments)
ml_client.batch_endpoints.begin_delete(name="endpoint-example").result()

# Azure AI Foundry

In [2]:
# connect 

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from openai import AzureOpenAI

try:
    
    # connect to the project
    project_endpoint = "https://......"
    project_client = AIProjectClient(            
            credential=DefaultAzureCredential(),
            endpoint=project_endpoint,
        )
    
    # Get a chat client
    chat_client = project_client.get_openai_client(api_version="2024-10-21")
    
    # Get a chat completion based on a user-provided prompt
    user_prompt = input("Enter a question:")
    
    response = chat_client.chat.completions.create(
        model=your_model_deployment_name,
        messages=[
            {"role": "system", "content": "You are a helpful AI assistant."},
            {"role": "user", "content": user_prompt}
        ]
    )
    print(response.choices[0].message.content)

except Exception as ex:
    print(ex)